In [ ]:
from utils.models.auto_anomaly.auto_anomaly import AutoAnomaly

model = AutoAnomaly(onnx_model_path="/workspace/src/utils/utils/models/auto_anomaly/assets/vit_base_patch16_dinov3.lvd1689m_464.onnx",
                    target_img_size=464)

model.fit_sample(
    images_path="/mnt/nas/SPA006/26_02_04/DA4214695",
    save_path="/workspace/src/SPA006/state.npz",
    n=3)

In [ ]:
from utils.models.auto_anomaly.auto_anomaly import AutoAnomaly
from utils.image_tools import random_image as ra
from utils.image_tools import visualizations as vi
from matplotlib import pyplot as plt
import time

times = []

model = AutoAnomaly(onnx_model_path="/workspace/src/utils/utils/models/auto_anomaly/assets/vit_base_patch16_dinov3.lvd1689m_464.onnx",
                    warmup_window_size=10, 
                    inference_window_size=15,
                    target_img_size=464,
                    area_threshold=5,
                    global_area_threshold=30,
                    anomaly_threshold=0.6,
                    start_gamma=0.4,
                    ramp_start=1,
                    ramp_end=20,
                    use_custom_min_max=True,
                    anomaly_min=140,
                    anomaly_max=800)

model.try_load_warmup("/workspace/src/SPA006")

for i in range(5):
    img, filename = ra.load_random_image("/mnt/nas/SPA006/26_02_04/DA4214695")
    start = time.perf_counter()
    overlay, mask, colored_map, discard = model.predict(img)
    if discard:
        anomaly = colored_map
    end = time.perf_counter()
    times.append(end - start)

avg_time = sum(times) / len(times)
print(f"Tempo medio di inferenza: {avg_time:.4f} secondi ({avg_time*1000:.2f} ms)")